# Φάση 1

## Βήμα 1: Προεπεξεργασία

### Φόρτωση συλλογής IR2025 (documents.csv)

In [1]:
# pip install pandas

In [2]:
import pandas as pd
docs = pd.read_csv("documents.csv")  # στήλες: ID, Text
print(docs)
print("Σύνολο εγγράφων:", len(docs))

           ID                                               Text
0      193157  Support towards the Europe PMC initiative-Cont...
1      193158  Support to the Vice-Presidents of the ERC Scie...
2      193159  Implementation of activities described in the ...
3      193160  Monitoring Atmospheric Composition and Climate...
4      193161  Pre-Operational Marine Service Continuity in T...
...       ...                                                ...
18311  215952  Better Rural Innovation: Linking Actors, Instr...
18312  215953  Pan-european assessment, monitoring, and mitig...
18313  215954  SMAll RuminanTs breeding for Efficiency and Re...
18314  215955  Networking, partnerships and tools to enhance ...
18315  215956  Towards Innovation - driven and smart solution...

[18316 rows x 2 columns]
Σύνολο εγγράφων: 18316


### Επειδή στην εργασία θα χρησιμοποιήσουμε τον english analyzer δεν χρειάζεται περεταίρω προεπεξεργασία

## Βήμα 2: Δημιουργία Ευρετηρίου στο ElasticSearch

### Εγκατάσταση Python client για Elasticsearch

In [3]:
# pip install elasticsearch

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

client = Elasticsearch("http://localhost:9200")
print(client.info())

{'name': 'LAPTOP', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'UDQcMKW7QPyBcJ4ghWklhQ', 'version': {'number': '9.2.1', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '4ad0ef0e98a2e72fafbd79a19fa5cae2f026117d', 'build_date': '2025-11-06T22:07:39.673130621Z', 'build_snapshot': False, 'lucene_version': '10.3.1', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


### Διαγραφή παλιού index (αν υπάρχει)

In [4]:
index_name = "ir2025_index"

# σβήνουμε το index αν υπάρχει ήδη
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

### Δημιουργία index με mapping (κλασικό BM25Similarity)

In [5]:
mapping = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "similarity": {
            "default": {
                "type": "BM25",
                "k1": 1.2,
                "b": 0.75
            }
        }
    },


    # χρησιμοποιούμε english analyzer
    
    "mappings": {
        "properties": {
            "id": {
                "type": "integer"
            },
            "text": {
                "type": "text",
                "analyzer": "english"
            }
        }
    }
}

client.indices.create(index=index_name, body=mapping)
print("Έτοιμο το index:", index_name)

Έτοιμο το index: ir2025_index


### Bulk indexing στο Elasticsearch

In [6]:
from elasticsearch.helpers import bulk

def generate_documents():
    """
    Παίρνει τα έγγραφα από το DataFrame docs
    και τα δίνει ένα-ένα στο bulk helper για indexing.
    """
    for _, row in docs.iterrows():
        doc_id = int(row["ID"])
        yield {
            "_index": index_name,
            "_id": doc_id,
            "_source": {
                "id": doc_id,
                "text": row["Text"]
            }
        }

# Εκτέλεση bulk indexing
success, errors = bulk(
    client,
    generate_documents(),   
    refresh=True,
    stats_only=True
)

print(f"Έγινε indexing {success} εγγράφων στο index '{index_name}'")
if errors:
    print("Υπήρξαν σφάλματα στο bulk:", errors)


Έγινε indexing 18316 εγγράφων στο index 'ir2025_index'


### Έλεγχος ότι όλα είναι μέσα

In [7]:
# Επιστρέφει πόσα έγγραφα υπάρχουν μέσα στο index
client.count(index=index_name)

ObjectApiResponse({'count': 18316, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

## Βήμα 3: Εκτέλεση ερωτημάτων & παραγωγή run files για το trec_eval

### Φόρτωση των queries (queries.csv)

In [8]:
import pandas as pd

# Φόρτωση ερωτημάτων
queries = pd.read_csv("queries.csv")   # στήλες: ID, Text
print("Σύνολο queries:",len(queries))
print(queries)


Σύνολο queries: 10
    ID                                               Text
0  Q01  EUTRAVEL Optimodal European Travel Ecosystem E...
1  Q02  Track And Know Big Data for Mobility Tracking ...
2  Q03  SELIS, Towards a Shared European Logistics Int...
3  Q04  TYPHON Polyglot and Hybrid Persistence Archite...
4  Q05  CHARIOT Cognitive Heterogeneous Architecture f...
5  Q06  MaaS4EU End-to-End Approach for Mobility-as-a-...
6  Q07  SOFIE SOFIE - Secure Open Federation for Inter...
7  Q08  DATASET2050 Data driven approach for a Seamles...
8  Q09  5GEx 5G Exchange The goal of the 5G Exchange (...
9  Q10  netCommons network infrastructure as commons C...


### Σύνδεση με ElasticSearch & ir2025_index

In [9]:
from elasticsearch import Elasticsearch

client = Elasticsearch("http://localhost:9200")
index_name = "ir2025_index"

### Συνάρτηση που τρέχει όλα τα queries και παράγει run files για το trec_eval

In [10]:
import os
RUNS_DIR = "runs_phase1"
os.makedirs(RUNS_DIR, exist_ok=True)

from elasticsearch.helpers import bulk

def run_all_queries_and_write_trec(k: int, run_name: str):
    """
    Τρέχει όλα τα queries από το queries DataFrame στο index `ir2025_index`
    και γράφει ένα TREC-style run file με top-k αποτελέσματα ανά query.
    """
    lines = []
    print("Θα τρέξω", len(queries), "queries με k =", k)
    for _, row in queries.iterrows():
        qid = row["ID"]              # π.χ. "Q01"
        qtext = row["Text"]    

        # Αναζήτηση στο ES με BM25
        res = client.search(
            index=index_name,
            query={"match": {"text": qtext}},
            size=k
        )

        rank = 1
        for hit in res["hits"]["hits"]:
            doc_id = hit["_source"]["id"]      # ID από documents.csv
            score = hit["_score"]

            # TREC format: qid Q0 docid rank score run_name
            line = f"{qid} Q0 {doc_id} {rank} {score:.4f} {run_name}"
            lines.append(line)
            rank += 1

    out_file = os.path.join(RUNS_DIR, f"bm25_k{k}_en.txt")
    with open(out_file, "w", encoding="utf-8") as f:
     f.write("\n".join(lines))

    print(f"Έγραψα το run file: {out_file} (k={k}, run_name={run_name})")


### Και εκτελούμε την συνάρτηση για τα τρία k: 20, 30, 50

In [11]:
run_all_queries_and_write_trec(20, "bm25_k20")
run_all_queries_and_write_trec(30, "bm25_k30")
run_all_queries_and_write_trec(50, "bm25_k50")

Θα τρέξω 10 queries με k = 20
Έγραψα το run file: runs_phase1\bm25_k20_en.txt (k=20, run_name=bm25_k20)
Θα τρέξω 10 queries με k = 30
Έγραψα το run file: runs_phase1\bm25_k30_en.txt (k=30, run_name=bm25_k30)
Θα τρέξω 10 queries με k = 50
Έγραψα το run file: runs_phase1\bm25_k50_en.txt (k=50, run_name=bm25_k50)
